# 05. Combining Data from Multiple Sources
**Engineering Data Processing with Python** | one-day course | about 30 minutes

> **How this notebook works**
> * Run every grey code cell in order: click it, then press **Shift + Enter**.
> * Most cells just need running and reading. Cells marked **Your turn** have a blank, `____`, for you to replace. The example just above shows you how.
> * Stuck? Click **Stuck? Click to show the answer** under the cell. Using it is fine.
> * Cells that show only a title are helper code written for you. Just run them.


In [ ]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

## The point of this module

Kilbrack's maintenance history is in two places: an old spreadsheet (2023 to March 2025) and the new maintenance
system, the CMMS (January 2025 onwards). During the switchover many jobs were recorded in **both**.
You will line up the two, combine them without double counting, and attach asset details.

Two operations:

| Operation | What it does | pandas |
|---|---|---|
| **Append** | Stack tables with the same columns: more rows | `pd.concat([a, b])` |
| **Join** | Add columns from another table, matched on a key: work order + its asset's type | `a.merge(b, on="asset_id")` |

In [ ]:
DATES = ["date_raised", "date_closed", "last_modified"]
cmms = pd.read_csv("data/checkpoints/m04_work_orders_dedup.csv", parse_dates=DATES)
cmms["priority"] = cmms["priority"].astype("Int64")
reg = pd.read_csv("data/checkpoints/m04_asset_register_dedup.csv")
tech_map = pd.read_csv("data/checkpoints/m04_technician_map.csv")
legacy_raw = pd.read_excel("data/work_orders_legacy.xlsx")
decom = pd.read_excel("data/asset_register.xlsx", sheet_name="Decommissioned")

print("CMMS columns:  ", cmms.columns.tolist()[:9])
print("Legacy columns:", legacy_raw.columns.tolist())

## 1. Line up the columns

Same information, different column names. A rename lookup fixes that.

### Your turn 5.1
Fill in the three missing new names. Compare with the CMMS column names printed above.

In [ ]:
LEGACY_COLUMNS = {
    "Job No": "wo_id",
    "Equipment": "____",
    "Job Type": "wo_type",
    "Raised": "____",
    "Completed": "date_closed",
    "Hrs": "____",
    "Cost (EUR)": "parts_cost_eur",
    "Fitter": "technician",
    "Notes": "description",
}
legacy = legacy_raw.rename(columns=LEGACY_COLUMNS)
legacy.head(3)

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
LEGACY_COLUMNS = {
    "Job No": "wo_id",
    "Equipment": "asset_id",
    "Job Type": "wo_type",
    "Raised": "date_raised",
    "Completed": "date_closed",
    "Hrs": "labour_hours",
    "Cost (EUR)": "parts_cost_eur",
    "Fitter": "technician",
    "Notes": "description",
}
legacy = legacy_raw.rename(columns=LEGACY_COLUMNS)
legacy.head(3)
```
</details>

In [ ]:
check({"asset_id", "date_raised", "labour_hours"} <= set(legacy.columns), "Columns lined up.",
      "Check the three names against the CMMS list.")

The rest of the alignment is written for you. Run and read the comments.

In [ ]:
# @title Helper: finish lining up the legacy table (just run it)
import unicodedata

def normalise_asset_id(series):
    # 'CNC001', 'cnc001', 'CNC-001 ' -> 'CNC-001'
    s = series.astype(str).str.strip().str.upper()
    return s.str.replace(r"^([A-Z]+)-?(\d+)$", r"\1-\2", regex=True)

def normalise_name(name):
    if pd.isna(name):
        return name
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    text = text.replace("'", "").replace(".", " ")
    if "," in text:
        last, first = [p.strip() for p in text.split(",", 1)]
        text = f"{first} {last}"
    return " ".join(text.lower().split())

legacy["wo_type"] = legacy["wo_type"].map({"Planned": "PM", "Breakdown": "CM", "Inspection": "INSP"})
legacy["asset_id"] = normalise_asset_id(legacy["asset_id"])
legacy["technician_raw"] = legacy["technician"]
legacy["technician"] = legacy["technician"].apply(normalise_name).map(dict(zip(tech_map["tech_norm"], tech_map["technician"])))
legacy["status"] = np.where(legacy["date_closed"].notna(), "Closed", "Open")   # a completed date means closed
legacy["priority"] = pd.Series(pd.NA, index=legacy.index, dtype="Int64")         # the old sheet had no priority: leave blank
legacy["last_modified"] = pd.NaT
legacy["hours_missing"] = legacy["labour_hours"].isna()
legacy = legacy[cmms.columns]                                                    # same columns, same order
print("Legacy now has the same columns as CMMS:", list(legacy.columns) == list(cmms.columns))

Notice: the old spreadsheet never recorded priority, so it stays **blank**. Lining up columns does not mean
inventing values.

## 2. Label where every row came from

When a number in a report looks odd, the first question is "where did this row come from?". Two extra columns
answer it. Example for the CMMS:

In [ ]:
cmms["source_system"] = "CMMS"
cmms["source_file"] = "work_orders_cmms.csv"

### Your turn 5.2
Do the same for the legacy table: system `LEGACY`, file `work_orders_legacy.xlsx`.

In [ ]:
legacy["source_system"] = "____"
legacy["source_file"] = "____"
from datetime import datetime
for df in (cmms, legacy):
    df["loaded_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
legacy["source_system"] = "LEGACY"
legacy["source_file"] = "work_orders_legacy.xlsx"
from datetime import datetime
for df in (cmms, legacy):
    df["loaded_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
```
</details>

## 3. Jobs recorded in both systems

The switchover jobs have **different IDs** in each system (`J10412` vs `WO-25-00031`), so we cannot match on ID.
Instead we match on what describes the job: **same asset, same day, same type**.

This part is written for you. The interesting bit is the safety check: `validate="one_to_one"` tells pandas
"each job should match at most one other job". Run and read.

In [ ]:
KEY = ["asset_id", "raised_day", "wo_type"]
for df in (cmms, legacy):
    df["raised_day"] = df["date_raised"].dt.normalize()     # date only, no time

try:
    legacy.merge(cmms[KEY], on=KEY, how="left", validate="one_to_one")
except Exception as err:
    print("pandas stopped us:", err)

cmms[cmms.duplicated(KEY, keep=False)][["wo_id", "asset_id", "date_raised", "wo_type", "description"]]

Good thing we asked. HYP-002 had two breakdowns on the same day. Without the check, each would match both, and two
jobs would become four rows without any warning. The fix numbers repeat jobs on the same day (1st, 2nd) and includes that in the
match. Run and read the result.

In [ ]:
cmms = cmms.sort_values("date_raised")
legacy = legacy.sort_values(["date_raised", "wo_id"])
for df in (cmms, legacy):
    df["seq"] = df.groupby(KEY).cumcount()

linked = legacy.merge(cmms[KEY + ["seq"]], on=KEY + ["seq"], how="left", validate="one_to_one", indicator=True)
in_both = linked[linked["_merge"] == "both"]
legacy_new = legacy[~legacy["wo_id"].isin(in_both["wo_id"])]

print("Legacy jobs already in the CMMS:", len(in_both))
print("Earliest of those:", in_both["date_raised"].min().date(), " latest:", in_both["date_raised"].max().date())
print("Legacy jobs to add:", len(legacy_new))

Every match falls inside the January to March 2025 switchover. That is evidence the matching works: if it had
matched jobs from 2023, it would be matching different jobs.

## 4. Append

### Your turn 5.3
Stack the CMMS jobs and the new legacy jobs. Choose from: `concat`, `merge`.

In [ ]:
combined = pd.____([cmms, legacy_new], ignore_index=True).drop(columns=["raised_day", "seq"])
print(len(cmms), "+", len(legacy_new), "=", len(combined))
combined["source_system"].value_counts()

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
combined = pd.concat([cmms, legacy_new], ignore_index=True).drop(columns=["raised_day", "seq"])
print(len(cmms), "+", len(legacy_new), "=", len(combined))
combined["source_system"].value_counts()
```
</details>

In [ ]:
check(len(combined) == len(cmms) + len(legacy_new) and combined["wo_id"].is_unique,
      "Combined, and every work order ID is still unique.", "Row count is off. Did you use concat?")

## 5. Join on asset details

Now add asset type and site to each work order. The asset list includes in-service and decommissioned assets.

In [ ]:
in_service = reg[["asset_id", "asset_type", "site", "criticality", "needs_review"]].assign(asset_status="In Service")
retired = (decom.rename(columns={"Asset ID": "asset_id", "Asset Type": "asset_type", "Site": "site", "Criticality": "criticality"})
                [["asset_id", "asset_type", "site", "criticality"]].assign(needs_review=False, asset_status="Decommissioned"))
assets_all = pd.concat([in_service, retired], ignore_index=True)
print(len(assets_all), "assets in the lookup")

### Your turn 5.4
`how="left"` keeps **every work order**, even ones whose asset is not in the list (those get blanks).
`how="inner"` keeps only work orders that found a match, and drops the rest without telling you.
We want to keep everything and see the problems. Which one?

In [ ]:
rows_before = len(combined)
wo_full = combined.merge(assets_all, on="asset_id", how="____", validate="many_to_one")
wo_full["asset_status"] = wo_full["asset_status"].fillna("Unknown asset")
print("Rows before:", rows_before, " after:", len(wo_full))
wo_full["asset_status"].value_counts()

<details><summary><b>Stuck? Click to show the answer</b></summary>

```python
rows_before = len(combined)
wo_full = combined.merge(assets_all, on="asset_id", how="left", validate="many_to_one")
wo_full["asset_status"] = wo_full["asset_status"].fillna("Unknown asset")
print("Rows before:", rows_before, " after:", len(wo_full))
wo_full["asset_status"].value_counts()
```
</details>

In [ ]:
check(len(wo_full) == rows_before and (wo_full["asset_status"] == "Unknown asset").sum() > 0,
      "No work orders lost, and the unknown assets are visible.",
      "Rows were lost. 'inner' drops unmatched work orders.")

**Always count rows before and after a join.** A join that changes the row count is the most common silent bug in
data work.

Look at the unknown assets:

In [ ]:
wo_full.loc[wo_full["asset_status"] == "Unknown asset", ["wo_id", "asset_id", "date_raised", "description"]]

`CMP-0O1` has a letter **O** instead of a zero. Impossible to spot by eye in Excel. Module 06 turns this into a
quality rule.

In [ ]:
wo_full.to_csv("outputs/work_orders_combined.csv", index=False)
print("Saved", wo_full.shape)

## Recap
* Line up columns with a rename lookup. Missing columns stay blank.
* Label every row with its source.
* No shared ID? Match on a combination of fields, and check the matches are where you expect.
* `validate=` and before/after row counts catch joins that go wrong.
* `how="left"` keeps everything so problems stay visible.

Next: **06 Data Quality Controls**.